# Pretrain Commutative CNN Encoder

Load the shared unlabeled pretraining dataset and save commutative CNN encoder weights for downstream classification notebooks.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from sklearn.model_selection import train_test_split

from src.ml import (
    CommutativeCNNClassifier,
    CommutativeCNNConfig,
    CommutativeCNNPretrainingConfig,
    LossWeightConfig,
    OptimizationConfig,
    augment_training_tensors_with_rotations,
    load_commutative_cnn_pretraining_config,
    write_commutative_cnn_pretraining_config,
)
from src.tensor_utils import load_unlabeled_tensor_dataset


In [2]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_cnn/encoder_state_v3.pt")
validation_fraction = 0.10
train_num_random_rotations = 0
rotation_range_degrees = 0.0

model_config = CommutativeCNNConfig(
    spatial_conv_channels=(8, 16),
    spatial_kernel_size_z=(3, 1),
    spatial_kernel_size_xy=(5, 3),
    spatial_stride_z=(1, 1),
    spatial_stride_xy=(1, 1),
    spatial_pool_kernel_z=(1, 1),
    spatial_pool_kernel_xy=(2, 2),
    spatial_pool_stride_z=(1, 1),
    spatial_pool_stride_xy=(2, 2),
    temporal_st_channels=(24,),
    temporal_st_kernel_sizes=(5,),
    temporal_ts_channels=(16, 24),
    temporal_ts_kernel_sizes=(7, 3),
    spatial_agg_channels=(16, 24),
    spatial_agg_kernel_size_z=(3, 1),
    spatial_agg_kernel_size_xy=(3, 3),
    spatial_agg_stride_z=(1, 1),
    spatial_agg_stride_xy=(1, 1),
    spatial_agg_pool_kernel_z=(1, 1),
    spatial_agg_pool_kernel_xy=(1, 2),
    spatial_agg_pool_stride_z=(1, 1),
    spatial_agg_pool_stride_xy=(1, 2),
    patch_size_z=1,
    patch_size_xy=16,
    embedding_dim=32,
    num_prototypes=32,
    dropout=0.1,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=40,
    learning_rate=8e-4,
    weight_decay=5e-5,
    early_stopping_patience=6,
    early_stopping_min_delta=1e-4,
    scheduler_patience=2,
    scheduler_factor=0.6,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    consistency_weight=1.0,
    feature_weight=0.05,
    prototype_temperature=0.2,
)

pretraining_config = CommutativeCNNPretrainingConfig(
    unlabeled_dataset_path=unlabeled_dataset_path,
    pretrained_encoder_path=pretrained_encoder_path,
    validation_fraction=validation_fraction,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
pretraining_config_path = write_commutative_cnn_pretraining_config(pretraining_config)
pretraining_config = load_commutative_cnn_pretraining_config(pretraining_config_path)
print(f"Loaded commutative CNN pretraining config from {pretraining_config_path}")
pretraining_config


Loaded commutative CNN pretraining config from /run/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/artifacts/pretrained_commutative_cnn/config.yaml


CommutativeCNNPretrainingConfig(unlabeled_dataset_path=PosixPath('.dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks'), pretrained_encoder_path=PosixPath('artifacts/pretrained_commutative_cnn/encoder_state_v3.pt'), validation_fraction=0.1, train_num_random_rotations=0, rotation_range_degrees=0.0, model_config=CommutativeCNNConfig(spatial_conv_channels=(8, 16), spatial_kernel_size_z=(3, 1), spatial_kernel_size_xy=(5, 3), spatial_stride_z=(1, 1), spatial_stride_xy=(1, 1), spatial_pool_kernel_z=(1, 1), spatial_pool_kernel_xy=(2, 2), spatial_pool_stride_z=(1, 1), spatial_pool_stride_xy=(2, 2), temporal_st_channels=(24,), temporal_st_kernel_sizes=(5,), temporal_ts_channels=(16, 24), temporal_ts_kernel_sizes=(7, 3), spatial_agg_channels=(16, 24), spatial_agg_kernel_size_z=(3, 1), spatial_agg_kernel_size_xy=(3, 3), spatial_agg_stride_z=(1, 1), spatial_agg_stride_xy=(1, 1), spatial_agg_pool_kernel_z=(1, 1), spatial_agg_pool_kernel_xy=(1, 2), spatial_agg_pool_stride_z=(1, 1), sp

In [3]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
train_indices, val_indices = train_test_split(
    range(len(unlabeled_dataset["tensors"])),
    test_size=validation_fraction,
    random_state=optimization_config.random_state,
    shuffle=True,
)
X_train_base = unlabeled_dataset["tensors"][train_indices]
X_val = unlabeled_dataset["tensors"][val_indices]
metadata_train_base = unlabeled_dataset["metadata"].iloc[train_indices].reset_index(drop=True)
metadata_val = unlabeled_dataset["metadata"].iloc[val_indices].reset_index(drop=True)
X_train, _, metadata_train = augment_training_tensors_with_rotations(
    X_train_base,
    [0] * len(X_train_base),
    metadata=metadata_train_base,
    num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)
{
    "all_tensors": unlabeled_dataset["tensors"].shape,
    "all_metadata": unlabeled_dataset["metadata"].shape,
    "train_base_tensors": X_train_base.shape,
    "train_tensors": X_train.shape,
    "val_tensors": X_val.shape,
    "train_base_metadata": metadata_train_base.shape,
    "train_metadata": metadata_train.shape,
    "val_metadata": metadata_val.shape,
}


{'all_tensors': torch.Size([2144, 20, 5, 96, 96]),
 'all_metadata': (2144, 7),
 'train_base_tensors': torch.Size([1929, 20, 5, 96, 96]),
 'train_tensors': torch.Size([1929, 20, 5, 96, 96]),
 'val_tensors': torch.Size([215, 20, 5, 96, 96]),
 'train_base_metadata': (1929, 7),
 'train_metadata': (1929, 7),
 'val_metadata': (215, 7)}

In [4]:
%%time
model = CommutativeCNNClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(X_train, validation_data=X_val)
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path


cols:
    ep=epoch
    lr=learning_rate
    eta=estimated_time_remaining
    trL=train_loss
    trCC=train_commutative_consistency_loss
    trFA=train_feature_alignment_loss
     ep       lr       eta |      trL     trCC     trFA |      vaL     vaCC     vaFA
001/040 8.00e-04  30:07:03 |   3.4209   3.4182   0.0528 |   3.4414   3.4397   0.0356
002/040 8.00e-04  29:20:06 |   3.4484   3.4475   0.0166 |   3.4492   3.4488   0.0082
003/040 8.00e-04  26:40:42 |   3.4534   3.4530   0.0084 |   3.4529   3.4525   0.0093
004/040 4.80e-04  24:43:20 |   3.4553   3.4550   0.0053 |   3.4546   3.4545   0.0020
005/040 4.80e-04  24:53:38 |   3.4560   3.4558   0.0039 |   3.4555   3.4555   0.0010
006/040 4.80e-04  23:59:40 |   3.4565   3.4564   0.0032 |   3.4562   3.4561   0.0007
007/040 2.88e-04  21:27:41 |   3.4569   3.4567   0.0028 |   3.4567   3.4566   0.0006
early_stop epoch=007 best_epoch=001 best_metric=3.4414
CPU times: user 3d 2h 49min 20s, sys: 1h 44min 53s, total: 3d 4h 34min 14s
Wall time: 4h 33

PosixPath('artifacts/pretrained_commutative_cnn/encoder_state_v3.pt')

In [5]:
model.pretrain_history_.tail()

,epoch,train_loss,train_commutative_consistency_loss,train_feature_alignment_loss,val_loss,val_commutative_consistency_loss,val_feature_alignment_loss
2,3,3.453392,3.452974,0.008360,3.452921,3.452456,0.009320
3,4,3.455257,3.454994,0.005259,3.454585,3.454485,0.001997
4,5,3.456041,3.455846,0.003899,3.455549,3.455498,0.001008
5,6,3.456517,3.456355,0.003231,3.456168,3.456132,0.000739
6,7,3.456883,3.456741,0.002829,3.456656,3.456625,0.000623
